In [1]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
import geopandas as gpd
import logging
from datetime import datetime

In [29]:
# 1. Logging Setup

logging.basicConfig(
    filename='spatial_join_automation.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 2. AGOL Login

try:
    gis = GIS("https://www.arcgis.com", "ForestryLAC", "Pacoima2024#")
    logging.info("Connected to AGOL successfully.")
except Exception as e:
    logging.error(f"Failed to connect to AGOL: {e}")
    raise

# 3. Layer URLs / Item IDs

PARCELS_URL = "https://services.arcgis.com/RmCCgQtiZLDCtblq/arcgis/rest/services/Topanga_Inspection_Point_Layer/FeatureServer/12"
POINTS_URL  = "https://services.arcgis.com/RmCCgQtiZLDCtblq/arcgis/rest/services/Topanga_Inspection_Point_Layer/FeatureServer/20"
OUTPUT_ITEM_ID = "f8cc45edc2594176a5f31c405e2d6c10" 

# 4. Load Layers

parcels_fl = FeatureLayer(PARCELS_URL)
points_fl  = FeatureLayer(POINTS_URL)

# Load layers with geometry
parcels_df = parcels_fl.query(where="1=1", as_df=True, return_geometry=True)
points_df  = points_fl.query(where="1=1", as_df=True, return_geometry=True)

# Convert to GeoDataFrame
parcels_gdf = gpd.GeoDataFrame(
    parcels_df,
    geometry="SHAPE",
    crs="EPSG:4326"
)

points_gdf = gpd.GeoDataFrame(
    points_df,
    geometry="SHAPE",
    crs="EPSG:4326"
)
#logging.info(f"Loaded {len(parcels_gdf)} parcels and {len(points_gdf)} points.")

# 5. Spatial Join

joined = gpd.sjoin(
    parcels_gdf,
    points_gdf,
    how="left",
    predicate="contains"
)

# 6. Count Points Per Parcel

counts = joined.groupby(joined.index).size()
parcels_gdf["Point_Count"] = counts
parcels_gdf["Point_Count"] = parcels_gdf["Point_Count"].fillna(0).astype(int)

# 7. Filter Parcels with Points Only

parcels_with_points = parcels_gdf[parcels_gdf["Point_Count"] > 0]

logging.info(f"{len(parcels_with_points)} parcels contain points.")

from arcgis.features import GeoAccessor

# 8. Convert GeoPandas GeoDataFrame to Spatially Enabled DataFrame
parcels_sedf = GeoAccessor.from_geodataframe(parcels_with_points)

print(type(parcels_sedf))
print(parcels_sedf.spatial.name)
print(parcels_sedf.spatial.validate(strict=True))

<class 'pandas.core.frame.DataFrame'>
SHAPE
True


In [30]:
import os
import tempfile
import arcpy

# Create a temporary directory
temp_dir = tempfile.mkdtemp()

# Define the GDB
gdb_name = "agol_overwrite.gdb"
gdb_path = os.path.join(temp_dir, gdb_name)

# Create the File Geodatabase
arcpy.management.CreateFileGDB(temp_dir, gdb_name)

<Result 'C:\\Users\\e705941\\AppData\\Local\\Temp\\tmpkgv2dgfd\\agol_overwrite.gdb'>

In [ ]:
import zipfile
from arcgis.features import FeatureLayerCollection

# --------------------------
# 1. Write SEDF to feature class
# --------------------------
fc_name = "parcels_with_points"  # name inside the GDB
fc_path = os.path.join(gdb_path, fc_name)

parcels_sedf.spatial.to_featureclass(location=fc_path, overwrite=True)
print(f"Feature class written to: {fc_path}")

# --------------------------
# 2. Zip the File Geodatabase
# --------------------------
zip_path = gdb_path + ".zip"

# Remove old zip if exists
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(gdb_path):
        for file in files:
            full_path = os.path.join(root, file)
            # store relative path inside zip
            arcname = os.path.relpath(full_path, os.path.dirname(gdb_path))
            zipf.write(full_path, arcname)

print(f"GDB zipped to: {zip_path}")
print("zip exists:", os.path.exists(zip_path))
print("zip is file:", os.path.isfile(zip_path))

# --------------------------
# 3. Overwrite the hosted AGOL layer
# --------------------------
output_item = gis.content.get(OUTPUT_ITEM_ID)
output_flc = FeatureLayerCollection.fromitem(output_item)

result = output_flc.manager.overwrite(zip_path)

if result.get("success"):
    print("✅ Hosted layer overwritten successfully.")
else:
    raise RuntimeError(f"Overwrite failed: {result}")